# Compile and deploy a model on NUCLEO-N657X0-Q

The reference mode reads the versioned COCO artifacts. For a custom model, use the TFLite and JSON from [Notebook 01](01_Train_and_Quantize_STM32N6.ipynb).

Install **ST Edge AI Core 3.0.0** to match the runtime in this project. Add `stedgeai` and `arm-none-eabi-objcopy` to PATH or set their absolute paths in the configuration cell. The Python cells stage files beside the repository. Follow the [README](../README.md#5-build-the-application) for firmware build and programming commands.

## What you will learn

Start with `USE_REFERENCE = True`. You can inspect a complete deployment bundle and calculate its flash span with no board or model compiler. Switch to `False` after finishing Notebook 01. A successful run then produces a new bundle and prints the C constants that need review before building the app.

The [architecture walkthrough](../Doc/Application-Overview.md) explains the two camera pipes, the generated tensor interface, and the division between linker-managed RAM and Neural-ART memory pools.

In [ ]:
from pathlib import Path
import json
import math
import re
import shutil
import subprocess
import tempfile

HERE = Path.cwd().resolve()
REPO = HERE if (HERE / "Model").is_dir() else HERE.parent
# REPO = Path(r"C:\projects\STM32N6-Edge_Computer_Vision")
MODEL_DIR = REPO / "Model"
BOARD = "NUCLEO-N657X0-Q"
BOARD_DIR = MODEL_DIR / BOARD
assert (BOARD_DIR / "stai_network.h").is_file(), "Set REPO to the repository root."
WORK = REPO.parent / "stm32n6_model_work"

# Reference mode uses the demo files. Custom mode builds files in a new folder.
USE_REFERENCE = True  # Set to False to compile your custom model.
MANIFEST = WORK / "insect_model.json"
TFLITE_OVERRIDE = None  # Optional: Path(r"C:\path\best_full_integer_quant.tflite")
STEDGEAI = "stedgeai"  # Or use the absolute path to stedgeai.exe.
OBJCOPY = "arm-none-eabi-objcopy"

## 1. Inspect the firmware's tensor interface

The YOLOv8 segmentation wrapper expects two signed 8-bit buffers. The first stores detections in attribute-first order. The second stores mask prototypes in height, width, channel order. With 80 classes, each candidate has 116 values (4 box coordinates, 80 class scores, and 32 mask coefficients). The 256 × 256 reference model produces 1,344 candidates and 32 mask prototypes of 64 × 64.

This notebook reads the interface that **ST Edge AI Core generates**. The header is the reference for the tensor shapes and quantization parameters used by the C application.

**Why read the generated header?** ST can change a tensor's interface type or reported shape during compilation. For example, the host TFLite uses signed RGB input, while the generated COCO input uses unsigned raw pixels. The C decoder also expects a specific output buffer order. Checking only the byte count misses errors where two layouts contain the same number of elements.

In [ ]:
def read_network_header(header_path):
    header = header_path.read_text(encoding="utf-8")
    # Join split C lines before reading the model settings.
    header = header.replace(chr(92) + chr(10), " ")

    def macro(suffix):
        match = re.search(r"^#define[ 	]+STAI_NETWORK_" + suffix + r"[ 	]+(.+)$", header, re.M)
        assert match, f"Missing {suffix} in {header_path}"
        return match.group(1).strip()

    # ST exposes the tensor format and affine parameters separately. Keep both for interface checks.
    def spec(prefix):
        return {
            "format": macro(prefix + "_FORMAT").strip("()"),
            "shape": [int(x) for x in macro(prefix + "_SHAPE").strip("{} ").split(",")],
            "scale": float(macro(prefix + "_SCALES").strip("{} ")),
            "zero_point": int(macro(prefix + "_OFFSETS").strip("{} ")),
        }

    return {
        "input": [spec(f"IN_{i}") for i in range(1, int(macro("IN_NUM").strip("()")) + 1)],
        "outputs": [spec(f"OUT_{i}") for i in range(1, int(macro("OUT_NUM").strip("()")) + 1)],
    }

baseline_specs = read_network_header(BOARD_DIR / "stai_network.h")
print(json.dumps(baseline_specs, indent=2))

## 2. Generate the model files

The repository [generation script](../Model/generate-n6-model_NUCLEO-N657X0-Q.sh) fixes the TFLite name to the COCO model. This notebook uses the same arguments and memory profile with a configurable input file. `--input-data-type uint8` matches the model interface to the RGB888 bytes produced by the camera. The profile reserves memory for the firmware and places the model data in external flash.

With `USE_REFERENCE = True`, this cell reads the existing files to show what a deployment bundle contains. With `False`, it runs `stedgeai generate` in a fresh directory, then runs `objcopy` to turn the raw xSPI data into an Intel HEX file starting at `0x70380000`. See the [manual commands](../README.md#3-generate-the-model-files) for the equivalent shell workflow.

| Setting | Why it matters |
| --- | --- |
| `--target stm32n6` | Selects the STM32N6 code generator and runtime interface. |
| `default@user_neuralart_….json` | Loads the named profile. The profile points to a board-specific memory pool, so keep that relative path when staging. |
| `--input-data-type uint8` | Requests unsigned input. The generated scale and zero point still have to match raw RGB. |
| `--change-addresses 0x70380000` | Gives the raw model bytes their external NOR address in the Intel HEX records. It must agree with the compiler's xSPI2 placement. |

If code generation succeeds but memory usage looks high, inspect each activation bank and software epoch in the report. The build map tells you about firmware memory. The model report tells you about the network placement.

In [ ]:
PROFILE = f"user_neuralart_{BOARD}.json"
MPOOL = f"stm32n6-app2_{BOARD}.mpool"
GENERATED_C = ("network.c", "network_ecblobs.h", "stai_network.c", "stai_network.h")

def run_checked(args, cwd):
    # Show the tool output and stop if a command fails.
    result = subprocess.run(args, cwd=cwd, capture_output=True, text=True, errors="replace")
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    result.check_returncode()

if USE_REFERENCE:
    GENERATED = BOARD_DIR
    NETWORK_HEX = BOARD_DIR / "network_data.hex"
    REPORT = MODEL_DIR / "st_ai_output" / "network_generate_report.txt"
    class_names = None
else:
    metadata = json.loads(MANIFEST.read_text(encoding="utf-8"))
    class_names = metadata["class_names"]
    assert class_names and len(class_names) <= 255 and metadata["imgsz"] == 256
    tflite = Path(TFLITE_OVERRIDE or metadata["model"]).resolve()
    assert tflite.is_file(), tflite

    WORK.mkdir(exist_ok=True)
    # Use a fresh folder so a failed build cannot pick up old model files.
    stage = Path(tempfile.mkdtemp(prefix="nucleo_", dir=WORK))
    (stage / "my_mpools").mkdir()
    # Run here so the settings file can find the memory layout file.
    shutil.copy2(MODEL_DIR / PROFILE, stage)
    shutil.copy2(MODEL_DIR / "my_mpools" / MPOOL, stage / "my_mpools")
    run_checked([
        STEDGEAI, "generate", "--model", str(tflite), "--target", "stm32n6",
        "--st-neural-art", f"default@{PROFILE}", "--input-data-type", "uint8",
    ], stage)
    GENERATED = stage / "st_ai_output"
    REPORT = GENERATED / "network_generate_report.txt"
    raw = GENERATED / "network_atonbuf.xSPI2.raw"
    assert raw.is_file() and raw.stat().st_size > 0, raw
    # Add the flash address so the programmer knows where to write the model data.
    NETWORK_HEX = stage / "network_data.hex"
    run_checked([
        OBJCOPY, "-I", "binary", str(raw), "--change-addresses",
        "0x70380000", "-O", "ihex", str(NETWORK_HEX),
    ], stage)

for name in GENERATED_C:
    assert (GENERATED / name).is_file(), GENERATED / name
assert NETWORK_HEX.is_file() and REPORT.is_file()
print("Generated code:", GENERATED)
print("Model data:", NETWORK_HEX)
print("Report:", REPORT)

## 3. Check the generated interface and update the firmware

Check output order, shapes, types, and quantization, then test the meaning of the outputs on a known image. The wrapper expects normalized box centers and dimensions, class scores, and mask coefficients. The header checks below cover the tensor structure. An image test is needed to establish that the decoder interprets the values correctly.

The detection and prototype buffers each have their own scale and zero point. The wrapper currently reads both pairs from `app_config.h`. Reusing the COCO values for a different model can distort boxes, scores, and masks.

The demo reads one scale and zero point per output from `app_config.h`. For example, COCO's detection tensor uses zero point 25 while the prototype tensor uses −115. After a model swap, inspect both pairs even if both tensors still use signed 8-bit values. Incorrect pairs can change the effective confidence threshold, box coordinates, and mask reconstruction.

In [ ]:
result_specs = read_network_header(GENERATED / "stai_network.h")
input_specs = result_specs["input"]
output_specs = result_specs["outputs"]
print(json.dumps(result_specs, indent=2))

assert len(input_specs) == 1 and len(output_specs) == 2
# The camera writes bytes directly. Scale 1/255 and zero point 0 make those bytes represent normalized RGB.
inp = input_specs[0]
assert inp["format"] == "STAI_FORMAT_U8" and inp["shape"] == [1, 256, 256, 3]
assert inp["zero_point"] == 0 and math.isclose(inp["scale"], 1 / 255, rel_tol=1e-5), (
    "The model input cannot use raw RGB bytes directly. Adjust preprocessing."
)
# The firmware reads detections first and mask templates second.
detections, prototypes = output_specs
class_count = 80 if USE_REFERENCE else len(class_names)
assert detections["shape"] == [4 + class_count + 32, 1344, 1]
assert prototypes["shape"] == [1, 64, 64, 32]
assert all(s["format"] == "STAI_FORMAT_S8" and s["scale"] > 0 for s in output_specs)

if class_names is not None:
    # Replace the corresponding definitions in Inc/app_config.h.
    print(f"#define NB_CLASSES ({class_count})")
    label_literals = [json.dumps(name, ensure_ascii=False) for name in class_names]
    labels = ", ".join(label_literals)
    print("#define CLASSES_TABLE", "const char* classes_table[NB_CLASSES] = {" + labels + "};")
print(f'#define AI_YOLOV8_SEG_ZERO_POINT ({detections["zero_point"]})')
print(f'#define AI_YOLOV8_SEG_SCALE ({detections["scale"]:.9f}f)')
print(f'#define AI_YOLOV8_SEG_MASK_ZERO_POINT ({prototypes["zero_point"]})')
print(f'#define AI_YOLOV8_SEG_MASK_SCALE ({prototypes["scale"]:.9f}f)')

# Read actual placements and CPU fallback information before changing the linker or memory pool.
print("\nMemory usage and execution placement:")
for line in REPORT.read_text(encoding="utf-8", errors="replace").splitlines():
    if any(key in line for key in ("used)", "Total number of epochs", "pure software", "hybrid epochs", "pure hardware")):
        print(line)

### Check where the model data will be written

Intel HEX stores byte records with addresses and checksums. The raw xSPI blob has no address field. For the repository's linear-address HEX, record type 04 supplies the upper 16 bits and type 00 adds its 16-bit offset and payload. Read those records to find the first and last occupied addresses before opening CubeProgrammer.

The first data byte must be at `0x70380000`. The last byte must fit in the NUCLEO's 64 MiB NOR, whose final address is `0x73FFFFFF`. The model compiler's memory-pool size is a separate limit.

In [ ]:
first_address = None
end_address = 0  # This is the first address after the model data.
upper_address = 0

with NETWORK_HEX.open(encoding="ascii") as hex_file:
    for line in hex_file:
        line = line.strip()
        assert line.startswith(":"), "Expected an Intel HEX record."
        record = bytes.fromhex(line[1:])
        count = record[0]
        assert len(record) == count + 5 and sum(record) % 256 == 0, "Invalid HEX record."
        offset = int.from_bytes(record[1:3], "big")
        record_type = record[3]
        if record_type == 4:
            upper_address = int.from_bytes(record[4:6], "big") << 16
        elif record_type == 0 and count:
            start = upper_address + offset
            first_address = start if first_address is None else min(first_address, start)
            end_address = max(end_address, start + count)

assert first_address == 0x70380000, f"Unexpected weights base: {first_address!r}"
assert end_address <= 0x74000000, "Model data extends past the physical external flash."
print(f"Model data spans {first_address:#010x} to {end_address - 1:#010x} "
      f"({(end_address - first_address) / 1024**2:.3f} MiB address span).")

## 4. Integrate, build, and flash

1. For a custom model, copy the staged `network.c`, `network_ecblobs.h`, `stai_network.c`, `stai_network.h`, and `network_data.hex` from the same generation into `Model/NUCLEO-N657X0-Q/`. Optionally copy the raw xSPI data as `network_data.xSPI2.bin`.
2. Replace the corresponding values in `Application/NUCLEO-N657X0-Q/Inc/app_config.h` using the cell output. Keep `POSTPROCESS_ISEG_YOLO_V8_UI` if the new model uses the same decoder. Check the generation report for memory usage and software fallbacks.
3. Clean and rebuild the UVCL application. The [README build section](../README.md#5-build-the-application) gives the exact CubeIDE post-build command and the Makefile alternative. The application image goes at `0x70100000`.
4. Follow the [README flash instructions](../README.md#6-program-external-flash) for the boot jumpers, external loader, and programming examples. The FSBL goes at `0x70000000`. The model HEX encodes `0x70380000`. When changing models, update the application image and its matching model HEX.

### Validate on the board

- Confirm that the console prints the expected model name.
- Test positive and empty scenes through CN8 USB video. For a controlled PC comparison, feed identical input RGB bytes and check the raw output tensors before applying the decoder.
- Record memory usage and latency. The demo timer measures the network call. Measure capture and postprocessing separately if you need end-to-end latency.

For an event-triggered imaging system, run inference after the ToF trigger and map the detection coordinates back to the original frame if they control a high-resolution crop.